# Importing Modules

In [2]:
!git clone https://github.com/Radhika-Amar-Desai/BSF_implementation.git

Cloning into 'BSF_implementation'...
remote: Enumerating objects: 266, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 266 (delta 2), reused 0 (delta 0), pack-reused 260 (from 2)
Receiving objects: 100% (266/266), 465.94 MiB | 20.04 MiB/s, done.
Resolving deltas: 100% (121/121), done.
Updating files: 100% (42/42), done.


In [3]:
!git clone https://github.com/facebookresearch/dinov3

Cloning into 'dinov3'...
remote: Enumerating objects: 660, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 660 (delta 8), reused 2 (delta 2), pack-reused 625 (from 3)
Receiving objects: 100% (660/660), 13.10 MiB | 19.31 MiB/s, done.
Resolving deltas: 100% (215/215), done.


In [4]:
import sys
from pathlib import Path

# Add the project root (parent of notebooks) to Python's search path
sys.path.append(str(Path().resolve().parent))

import os
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

# Configuration

In [5]:
# ACTIVATION_FOLDER = "data/rabit_activations_dinov3"
INPUT_DIM = 768
NUM_BLOCKS = 256
BLOCK_SIZE = 3
TOP_K = 8
BATCH_SIZE = 2048
LR = 3e-3
EPOCHS = 300
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Generating Activations

In [6]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 60.7 MB/s eta 0:00:00


In [7]:
!pip install einops

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import sys
import pathlib
import numpy as np
import torch
from torchvision.transforms import v2
import einops

POS_MEAN_PATH = '/content/BSF_implementation/pos_mean.npy'
# (n_patches, d)
POS_MEAN = np.load(POS_MEAN_PATH).astype(np.float32)

# Local repo settings
DINO_REPO = "/content/dinov3"
WEIGHTS = "/content/drive/MyDrive/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"

# 1. FIX: Prepend the repository path so that hubconf.py can resolve internal modules
if DINO_REPO not in sys.path:
    sys.path.insert(0, DINO_REPO)

def load_rabbit_images(npz_path):
    """Load the rabbit images as a (N, H, W, 3) uint8 array."""
    return np.load(npz_path)['arr_0']

@torch.no_grad()
def dino_activations(
    images,
    *,
    device=None,
    batch_size=64,
):
    """
    Parameters
    ----------
    images : ndarray
        (N,H,W,3) uint8

    Returns
    -------
    ndarray
        (N,196,768) float32 patch embeddings
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    if not pathlib.Path(WEIGHTS).exists():
        raise FileNotFoundError(
            f"Model weights not found at {WEIGHTS}. Please ensure the weights file "
            "'dinov3_vitb16_pretrain_lvd1689m.pth' is downloaded and placed in your Google Drive."
        )

    # 2. FIX: Initialize the empty backbone via Hub, then manually inject the state dict.
    # DINOv3's hubconf models do not directly accept a 'weights' string path argument.
    model = torch.hub.load(
        DINO_REPO,
        "dinov3_vitb16",
        source="local",
        pretrained=False  # Tells hubconf not to fetch from remote Meta servers
    )

    # Load and map the local pth checkpoint weights safely
    state_dict = torch.load(WEIGHTS, map_location="cpu")

    # Unwrap 'model' or 'teacher' key if saved from a training checkpoint wrapper
    if "model" in state_dict:
        state_dict = state_dict["model"]
    elif "teacher" in state_dict:
        state_dict = state_dict["teacher"]

    model.load_state_dict(state_dict, strict=True)
    model = model.to(device).eval()

    transform = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        ),
    ])

    outputs = []

    for start in range(0, len(images), batch_size):
        batch = torch.stack(
            [transform(im) for im in images[start:start + batch_size]]
        ).to(device)

        feats = model.forward_features(batch)

        # Extract features (B, 196, 768) for ViT-B/16 @ 224x224
        patch_tokens = feats["x_norm_patchtokens"]
        outputs.append(patch_tokens.cpu())

    return torch.cat(outputs, dim=0).numpy()

def patch_grid(n_patches):
    """Side length of the square patch grid (14 for DINOv3 ViT-B/16 @ 224)."""
    g = int(round(n_patches ** 0.5))
    assert g * g == n_patches, f'{n_patches} patches is not a square grid'
    return g

# Rabbit images -> DINOv3 patch activations (fp32). Needs a GPU; ~1 min.
images = load_rabbit_images('/content/BSF_implementation/data/rabbit.npz')
acts = dino_activations(images)

# centre by the ImageNet per-position mean, flatten, scale so ||x|| ~ sqrt(d)
acts = acts - POS_MEAN
x = einops.rearrange(acts, 'n p d -> (n p) d')
x = x / np.sqrt((x ** 2).sum(1).mean()) * np.sqrt(x.shape[1])
grid = patch_grid(acts.shape[1])
print('activations:', x.shape, '  patch grid:', grid)

activations: (58800, 768)   patch grid: 14


# Dataset

In [11]:
class PatchDataset(Dataset):

    def __init__(self, activations):

        # Directly use the provided activations instead of loading from a folder
        self.samples = activations

        print("PatchDataset initialized with shape:", self.samples.shape)

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, idx):

        x = self.samples[idx]

        return torch.from_numpy(x).float()

# DataLoader

In [12]:
# Use the generated activations 'x' directly for the PatchDataset
dataset = PatchDataset(x)

train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)

PatchDataset initialized with shape: (58800, 768)


# MDL Formula

In [26]:
import math

import torch
import torch.nn as nn


class MDL(nn.Module):
    """
    Minimum Description Length (Eq. 5)

    Implements the three MDL terms used for evaluation:

        L = L_support + L_code + L_residual

    Dictionary cost is omitted.
    """

    def __init__(
        self,
        num_blocks,
        block_size,
        distortion,
    ):
        super().__init__()

        self.G = num_blocks
        self.b = block_size
        self.delta = distortion

    ###########################################################
    # log2(n choose k)
    ###########################################################

    @staticmethod
    def log2_binomial(n, k):

        if k < 0 or k > n:
            return 0.0

        return (
            math.lgamma(n + 1)
            - math.lgamma(k + 1)
            - math.lgamma(n - k + 1)
        ) / math.log(2)

    ###########################################################
    # Support term
    ###########################################################

    def support_bits(self, active_mask):

        k = active_mask.sum(dim=1)

        bits = torch.tensor(
            [
                self.log2_binomial(self.G, int(kk))
                for kk in k.tolist()
            ],
            device=active_mask.device,
            dtype=torch.float32,
        )

        # Average support cost over the dataset
        return bits.mean()

    ###########################################################
    # Code term
    ###########################################################

    def code_bits(
        self,
        z_blocks,
        active_mask,
    ):

        total_bits = torch.tensor(
            0.0,
            device=z_blocks.device,
        )

        for g in range(self.G):

            mask = active_mask[:, g]

            n_active = mask.sum()

            if n_active <= 1:
                continue

            block = z_blocks[mask, g]

            p_g = mask.float().mean()

            block = (
                block
                - block.mean(
                    dim=0,
                    keepdim=True,
                )
            )

            cov = (
                block.T @ block
            ) / (n_active - 1)

            eigvals = torch.linalg.eigvalsh(cov)

            eigvals = torch.clamp(
                eigvals,
                min=0,
            )

            block_bits = (
                0.5
                * torch.log2(
                    1 + eigvals / self.delta
                )
            ).sum()

            total_bits += p_g * block_bits

        return total_bits

    ###########################################################
    # Residual term
    ###########################################################

    def residual_bits(
        self,
        x,
        reconstruction,
    ):

        residual = x - reconstruction

        residual = (
            residual
            - residual.mean(
                dim=0,
                keepdim=True,
            )
        )

        cov = (
            residual.T @ residual
        ) / (residual.shape[0] - 1)

        eigvals = torch.linalg.eigvalsh(cov)

        eigvals = torch.clamp(
            eigvals,
            min=0,
        )

        bits = (
            0.5
            * torch.log2(
                1 + eigvals / self.delta
            )
        ).sum()

        return bits

    ###########################################################
    # Total MDL
    ###########################################################

    def forward(
        self,
        x,
        output,
    ):

        z_blocks = output["z_blocks"]

        block_norms = torch.norm(
            z_blocks,
            dim=-1,
        )

        active_mask = block_norms > 0

        support = self.support_bits(
            active_mask,
        )

        code = self.code_bits(
            z_blocks,
            active_mask,
        )

        residual = self.residual_bits(
            x,
            output["reconstruction"],
        )

        total = (
            support
            + code
            + residual
        )

        return {
            "total": total,
            "support": support,
            "code": code,
            "residual": residual,
        }

# MDL For Grassmann BSF

In [31]:
import os
import pathlib
import torch

os.chdir("/content/BSF_implementation")

from models.improved_grassmann_bsf import build_model

####################################################
# Load Model
####################################################

model = build_model(
    input_dim=INPUT_DIM,
    num_blocks=NUM_BLOCKS,
    block_size=BLOCK_SIZE,
    top_k=TOP_K,
)

BSF_WEIGHTS_PATH = (
    "/content/BSF_implementation/model_checkpoints/"
    "improved_grassmann_bsf_dinov3.pth"
)

if not pathlib.Path(BSF_WEIGHTS_PATH).exists():
    raise FileNotFoundError(
        f"BSF model weights not found at {BSF_WEIGHTS_PATH}"
    )

state_dict = torch.load(
    BSF_WEIGHTS_PATH,
    map_location="cpu",
)

model.load_state_dict(state_dict)
model.to(DEVICE).eval()

print(f"Loaded weights from {BSF_WEIGHTS_PATH}")

####################################################
# MDL
####################################################

# Distortion level δ used in the paper.
# Typical choices: 0.01, 0.05, 0.10, 0.20
DISTORTION = 0.10

mdl_calculator = MDL(
    num_blocks=model.num_blocks,
    block_size=model.block_size,
    distortion=DISTORTION,
)

####################################################
# Collect entire evaluation set
####################################################

all_x = []
all_reconstruction = []
all_z_blocks = []

rss = 0.0
tss = 0.0

with torch.no_grad():

    for x in train_loader:

        x = x.to(DEVICE)

        output = model(x)

        reconstruction = output["reconstruction"]

        ########################################
        # Save everything for MDL
        ########################################

        all_x.append(x.cpu())
        all_reconstruction.append(reconstruction.cpu())
        all_z_blocks.append(output["z_blocks"].cpu())

        ########################################
        # Reconstruction statistics
        ########################################

        residual = x - reconstruction

        rss += residual.pow(2).sum().item()

        tss += (
            (x - x.mean(dim=0, keepdim=True))
            .pow(2)
            .sum()
            .item()
        )

####################################################
# Concatenate entire dataset
####################################################

all_x = torch.cat(all_x, dim=0).to(DEVICE)

all_reconstruction = torch.cat(
    all_reconstruction,
    dim=0,
).to(DEVICE)

all_z_blocks = torch.cat(
    all_z_blocks,
    dim=0,
).to(DEVICE)

####################################################
# Compute MDL once over the entire dataset
####################################################

mdl_output = mdl_calculator(
    all_x,
    {
        "reconstruction": all_reconstruction,
        "z_blocks": all_z_blocks,
    },
)

####################################################
# Reconstruction quality
####################################################

r2 = 1 - rss / tss

####################################################
# Convert tensors to Python scalars
####################################################

support_bits = mdl_output["support"].item()
code_bits = mdl_output["code"].item()
residual_bits = mdl_output["residual"].item()
total_bits = mdl_output["total"].item()

####################################################
# Results
####################################################

print(f"Support Bits        : {support_bits:.3f}")
print(f"Code Bits           : {code_bits:.3f}")
print(f"Residual Bits       : {residual_bits:.3f}")
print("-" * 45)
print(f"Total MDL           : {total_bits:.3f} bits")
print(f"Reconstruction R²   : {r2:.4f}")

Loaded weights from /content/BSF_implementation/model_checkpoints/improved_grassmann_bsf_dinov3.pth
Support Bits        : 48.541
Code Bits           : 78.936
Residual Bits       : 456.527
---------------------------------------------
Total MDL           : 584.005 bits
Reconstruction R²   : 0.8011


# MDL for Top-k SAE

In [32]:
import pathlib
import torch

from models.top_k_SAE import build_model

####################################################
# Load Model
####################################################

model = build_model(
    input_dim=INPUT_DIM,
    num_blocks=NUM_BLOCKS * BLOCK_SIZE,
    block_size=1,
    top_k=TOP_K * BLOCK_SIZE,
)

SAE_WEIGHTS_PATH = (
    "/content/BSF_implementation/model_checkpoints/"
    "top_k_sae_model.pth"
)

if not pathlib.Path(SAE_WEIGHTS_PATH).exists():
    raise FileNotFoundError(
        f"SAE model weights not found at {SAE_WEIGHTS_PATH}"
    )

state_dict = torch.load(
    SAE_WEIGHTS_PATH,
    map_location="cpu",
)

model.load_state_dict(state_dict)
model.to(DEVICE).eval()

print(f"Loaded weights from {SAE_WEIGHTS_PATH}")

####################################################
# MDL
####################################################

DISTORTION = 0.10

mdl_calculator = MDL(
    num_blocks=model.num_blocks,
    block_size=model.block_size,
    distortion=DISTORTION,
)

####################################################
# Collect entire evaluation set
####################################################

all_x = []
all_reconstruction = []
all_z_blocks = []

rss = 0.0
tss = 0.0

with torch.no_grad():

    for x in train_loader:

        x = x.to(DEVICE)

        output = model(x)

        reconstruction = output["reconstruction"]

        ########################################
        # Adapt SAE output for MDL
        ########################################

        z_blocks = output["z"].view(
            x.size(0),
            model.num_blocks,
            model.block_size,
        )

        ########################################
        # Save for dataset-level MDL
        ########################################

        all_x.append(x.cpu())

        all_reconstruction.append(
            reconstruction.cpu()
        )

        all_z_blocks.append(
            z_blocks.cpu()
        )

        ########################################
        # Reconstruction statistics
        ########################################

        residual = x - reconstruction

        rss += residual.pow(2).sum().item()

        tss += (
            (x - x.mean(dim=0, keepdim=True))
            .pow(2)
            .sum()
            .item()
        )

####################################################
# Concatenate entire dataset
####################################################

all_x = torch.cat(
    all_x,
    dim=0,
).to(DEVICE)

all_reconstruction = torch.cat(
    all_reconstruction,
    dim=0,
).to(DEVICE)

all_z_blocks = torch.cat(
    all_z_blocks,
    dim=0,
).to(DEVICE)

####################################################
# Compute MDL once over the entire dataset
####################################################

mdl_output = mdl_calculator(
    all_x,
    {
        "reconstruction": all_reconstruction,
        "z_blocks": all_z_blocks,
    },
)

####################################################
# Reconstruction quality
####################################################

r2 = 1 - rss / tss

####################################################
# Convert tensors to scalars
####################################################

support_bits = mdl_output["support"].item()
code_bits = mdl_output["code"].item()
residual_bits = mdl_output["residual"].item()
total_bits = mdl_output["total"].item()

####################################################
# Results
####################################################

print(f"Support Bits        : {support_bits:.3f}")
print(f"Code Bits           : {code_bits:.3f}")
print(f"Residual Bits       : {residual_bits:.3f}")
print("-" * 45)
print(f"Total MDL           : {total_bits:.3f} bits")
print(f"Reconstruction R²   : {r2:.4f}")

Loaded weights from /content/BSF_implementation/model_checkpoints/top_k_sae_model.pth
Support Bits        : 150.478
Code Bits           : 34.807
Residual Bits       : 417.943
---------------------------------------------
Total MDL           : 603.227 bits
Reconstruction R²   : 0.8357


# MDL for Vanilla BSF

In [34]:
import pathlib
import torch

from models.vanilla_bsf import build_model

####################################################
# Load Model
####################################################

model = build_model(
    input_dim=INPUT_DIM,
    num_blocks=NUM_BLOCKS,
    block_size=BLOCK_SIZE,
    top_k=TOP_K,
)

BSF_WEIGHTS_PATH = (
    "/content/BSF_implementation/model_checkpoints/"
    "improved_vanilla_bsf_dinov3.pth"
)

if not pathlib.Path(BSF_WEIGHTS_PATH).exists():
    raise FileNotFoundError(
        f"BSF model weights not found at {BSF_WEIGHTS_PATH}"
    )

state_dict = torch.load(
    BSF_WEIGHTS_PATH,
    map_location="cpu",
)

model.load_state_dict(state_dict)
model.to(DEVICE).eval()

print(f"Loaded weights from {BSF_WEIGHTS_PATH}")

####################################################
# MDL
####################################################

# Distortion level δ
DISTORTION = 0.10

mdl_calculator = MDL(
    num_blocks=NUM_BLOCKS,
    block_size=model.block_size,
    distortion=DISTORTION,
)

####################################################
# Collect entire evaluation set
####################################################

all_x = []
all_reconstruction = []
all_z_blocks = []

rss = 0.0
tss = 0.0

with torch.no_grad():

    for x in train_loader:

        x = x.to(DEVICE)

        output = model(x)

        reconstruction = output["reconstruction"]

        ########################################
        # Save everything for MDL
        ########################################

        all_x.append(x.cpu())

        all_reconstruction.append(
            reconstruction.cpu()
        )

        all_z_blocks.append(
            output["z_blocks"].cpu()
        )

        ########################################
        # Reconstruction statistics
        ########################################

        residual = x - reconstruction

        rss += residual.pow(2).sum().item()

        tss += (
            (x - x.mean(dim=0, keepdim=True))
            .pow(2)
            .sum()
            .item()
        )

####################################################
# Concatenate entire dataset
####################################################

all_x = torch.cat(
    all_x,
    dim=0,
).to(DEVICE)

all_reconstruction = torch.cat(
    all_reconstruction,
    dim=0,
).to(DEVICE)

all_z_blocks = torch.cat(
    all_z_blocks,
    dim=0,
).to(DEVICE)

####################################################
# Compute MDL once over the entire dataset
####################################################

mdl_output = mdl_calculator(
    all_x,
    {
        "reconstruction": all_reconstruction,
        "z_blocks": all_z_blocks,
    },
)

####################################################
# Reconstruction quality
####################################################

r2 = 1 - rss / tss

####################################################
# Convert tensors to scalars
####################################################

support_bits = mdl_output["support"].item()
code_bits = mdl_output["code"].item()
residual_bits = mdl_output["residual"].item()
total_bits = mdl_output["total"].item()

####################################################
# Results
####################################################

print(f"Support Bits        : {support_bits:.3f}")
print(f"Code Bits           : {code_bits:.3f}")
print(f"Residual Bits       : {residual_bits:.3f}")
print("-" * 45)
print(f"Total MDL           : {total_bits:.3f} bits")
print(f"Reconstruction R²   : {r2:.4f}")

Loaded weights from /content/BSF_implementation/model_checkpoints/improved_vanilla_bsf_dinov3.pth
Support Bits        : 48.541
Code Bits           : 59.434
Residual Bits       : 423.296
---------------------------------------------
Total MDL           : 531.272 bits
Reconstruction R²   : 0.8228


# MDL for Group Lasso BSF

In [35]:
import pathlib
import torch

from models.group_lasso_bsf import build_model

####################################################
# Load Model
####################################################

model = build_model(
    input_dim=INPUT_DIM,
    num_blocks=NUM_BLOCKS,
    block_size=BLOCK_SIZE,
)

BSF_WEIGHTS_PATH = (
    "/content/BSF_implementation/model_checkpoints/"
    "group_lasso_bsf_dinov3.pth"
)

if not pathlib.Path(BSF_WEIGHTS_PATH).exists():
    raise FileNotFoundError(
        f"BSF model weights not found at {BSF_WEIGHTS_PATH}"
    )

state_dict = torch.load(
    BSF_WEIGHTS_PATH,
    map_location="cpu",
)

model.load_state_dict(state_dict)
model.to(DEVICE).eval()

print(f"Loaded weights from {BSF_WEIGHTS_PATH}")

####################################################
# MDL
####################################################

# Distortion level δ
DISTORTION = 0.10

mdl_calculator = MDL(
    num_blocks=model.num_blocks,
    block_size=model.block_size,
    distortion=DISTORTION,
)

####################################################
# Collect entire evaluation set
####################################################

all_x = []
all_reconstruction = []
all_z_blocks = []

rss = 0.0
tss = 0.0

with torch.no_grad():

    for x in train_loader:

        x = x.to(DEVICE)

        output = model(x)

        reconstruction = output["reconstruction"]

        ########################################
        # Save everything for MDL
        ########################################

        all_x.append(x.cpu())

        all_reconstruction.append(
            reconstruction.cpu()
        )

        all_z_blocks.append(
            output["z_blocks"].cpu()
        )

        ########################################
        # Reconstruction statistics
        ########################################

        residual = x - reconstruction

        rss += residual.pow(2).sum().item()

        tss += (
            (x - x.mean(dim=0, keepdim=True))
            .pow(2)
            .sum()
            .item()
        )

####################################################
# Concatenate entire dataset
####################################################

all_x = torch.cat(
    all_x,
    dim=0,
).to(DEVICE)

all_reconstruction = torch.cat(
    all_reconstruction,
    dim=0,
).to(DEVICE)

all_z_blocks = torch.cat(
    all_z_blocks,
    dim=0,
).to(DEVICE)

####################################################
# Compute MDL once over the entire dataset
####################################################

mdl_output = mdl_calculator(
    all_x,
    {
        "reconstruction": all_reconstruction,
        "z_blocks": all_z_blocks,
    },
)

####################################################
# Reconstruction quality
####################################################

r2 = 1 - rss / tss

####################################################
# Convert tensors to scalars
####################################################

support_bits = mdl_output["support"].item()
code_bits = mdl_output["code"].item()
residual_bits = mdl_output["residual"].item()
total_bits = mdl_output["total"].item()

####################################################
# Results
####################################################

print(f"Support Bits        : {support_bits:.3f}")
print(f"Code Bits           : {code_bits:.3f}")
print(f"Residual Bits       : {residual_bits:.3f}")
print("-" * 45)
print(f"Total MDL           : {total_bits:.3f} bits")
print(f"Reconstruction R²   : {r2:.4f}")

Loaded weights from /content/BSF_implementation/model_checkpoints/group_lasso_bsf_dinov3.pth
Support Bits        : 206.816
Code Bits           : 125.686
Residual Bits       : 140.524
---------------------------------------------
Total MDL           : 473.026 bits
Reconstruction R²   : 0.9611
